<a href="https://colab.research.google.com/github/lekha-8/E-commerce-/blob/main/Text_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import textwrap
import re

# Load T5 model and tokenizer
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def capitalize_sentences(text):
    # Capitalize the first letter of each sentence after punctuation or start of string
    def cap(match):
        return match.group(1) + match.group(2).upper()
    return re.sub(r'(^|(?<=[\.\!\?]\s))([a-z])', cap, text)

def trim_summary(text, max_sentences=2):
    # Split text into sentences and keep max_sentences
    sentences = re.split(r'(?<=[.!?]) +', text)
    trimmed = ' '.join(sentences[:max_sentences])
    return trimmed.strip()

def summarize_abstract(paragraph, max_len=40, min_len=20):
    if not paragraph.strip():
        return "Empty input. Please enter a valid paragraph."

    # Prepare prompt for T5
    prompt = "summarize: " + paragraph.replace("\n", " ").strip()

    input_ids = tokenizer.encode(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)

    # Generate summary with parameters to reduce repetition and encourage conciseness
    summary_ids = model.generate(
        input_ids,
        max_length=max_len,
        min_length=min_len,
        length_penalty=3.0,          # higher penalty for longer outputs
        num_beams=5,                 # beam search for quality
        repetition_penalty=3.0,      # penalize repetition
        no_repeat_ngram_size=3,      # avoid repeating 3-grams
        early_stopping=True
    )

    raw_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    trimmed = trim_summary(raw_summary, max_sentences=2)
    return capitalize_sentences(trimmed)

def main():
    print("=== Abstract Overall Summary using T5 ===\n")
    print("Enter your paragraph (press Enter twice to finish):")

    lines = []
    while True:
        line = input()
        if line.strip() == "":
            break
        lines.append(line)
    paragraph = "\n".join(lines).strip()

    if not paragraph:
        print("No input provided. Exiting.")
        return

    print("\nOriginal Paragraph:\n")
    print(textwrap.fill(paragraph, width=80))

    summary = summarize_abstract(paragraph)

    print("\nOverall Abstract Summary (max 2 sentences):\n")
    print(textwrap.fill(summary, width=80))

if __name__ == "__main__":
    main()


=== Abstract Overall Summary using T5 ===

Enter your paragraph (press Enter twice to finish):
Regular physical activity is essential for maintaining good health and preventing chronic diseases such as heart disease, diabetes, and obesity. Engaging in at least 150 minutes of moderate-intensity exercise per week can improve cardiovascular health, strengthen muscles and bones, and boost mental well-being. Additionally, a balanced diet rich in fruits, vegetables, whole grains, and lean proteins supports the immune system and helps maintain a healthy weight. Preventive measures such as regular medical check-ups, vaccinations, and avoiding tobacco and excessive alcohol consumption are also critical components of a healthy lifestyle. By adopting these habits, individuals can enhance their quality of life and reduce the risk of developing serious health conditions.


Original Paragraph:

Regular physical activity is essential for maintaining good health and
preventing chronic diseases such as